### import

In [33]:
import pandas as pd
import numpy as np
import re, os, functions, warnings
from os.path import join
from typing import Optional
warnings.simplefilter("ignore")

deposit_dtype = {'채무자키':str, '입금키':str, '계좌키':str, '계좌번호':str, '입금고정키':str, '타채무자키':str}

# 출력 옵션
pd.options.display.float_format = '{:,.0f}'.format

# dtype 정의
debt_dtype = {'채무자키':str, '타채무자키':str, '담당자키':str, '관리자비고':str}
account_dtype = {'채무자키':str, '계좌키':str, '타채무자키':str}
grt_dtype = {'채무자키':str, '계좌키':str, '타채무자키':str, '보증인키':str}
event_dtype = {'채무자키':str, '법조치키':str, '계좌키':str, '관련법조치키':str, '법취하키':str, '타법조치키':str, '타채무자키':str, '관할법원코드':str}

def fill_groupwise(df:pd.DataFrame, column:str, group_cols: Optional[list[str]] = None) -> pd.DataFrame :
    """새채무자키와 사건번호(default)를 group_cols로하여 column에서 빈값을 동일 그룹의 앞뒤값으로 채워주는 함수
    column의 빈문자열을 na로 바꿔주는 코드 내포함"""
    if group_cols is None : group_cols = ["새채무자키", "사건번호"]
    df[column] = df[column].replace("", pd.NA)
    df[column] = (
        df.groupby(group_cols)[column]
        .transform(lambda x: x.ffill().bfill())
    )
    return df

### 법조치 사건정리

In [34]:
##################################
company = "솔림"      # 솔림 or 대성
basedate = "260531" 
##################################
path_write_dir = r"D:\3.자산\금감원\비정기 요청자료\260520"
if company == "대성" : 
    path_write_dir = join(path_write_dir, "대성")
    
기준일 = '20' + basedate[:2] +'-'+basedate[2:4]+'-'+basedate[4:6]
기준일_dt = pd.to_datetime(기준일)

wd = join(r"D:\3.자산\전산 dataset", company)


# 기본 경로
parquet_dir = join(wd, basedate, "parquet")

In [ ]:
# 파일 읽기
# 법조치 읽기
court_ori = pd.read_parquet(functions.키워드로파일명찾기(parquet_dir, "법조치조회",전체경로=True))

In [35]:
# 키리스트
keys = pd.read_excel(join(path_write_dir, "2.키리스트.xlsx"), dtype=str)

# 새채무자키
if company == "솔림" : 
    새채무자키 = pd.read_pickle(join(wd, r"D:\3.자산\POOL별 관리자산\솔림\전체", "새채무자키.pkl"))
else : 
    새채무자키 = pd.read_excel(join(wd, r"D:\3.자산\POOL별 관리자산\대성", "새채무자키.xlsx"), dtype=str)
    
새채무자키 = 새채무자키.drop_duplicates(subset=["채무자키"])[["새채무자키","채무자키"]]

In [ ]:
# court와 keys_unique를 채무자키로 merge
court_새채무자키 = court_ori.copy().merge(새채무자키, on="채무자키", how="left")

In [ ]:
법조치cols = ["새채무자키","채무자키","법조치키","법조치구분","법조치세부","관할법원","사건번호","사건구분","대상자구분","대상자명",
           "접수일","종료일","종국결과일","확정일","종국결과","진행결과","비고","법조치메모",
           "관련사건법원","관련사건번호","최신화결과화면","최신화일"]
# court_새채무자키에서 새채무자키가 keys의 새채무자키에 있는 것만 남기기
court = court_새채무자키[court_새채무자키["새채무자키"].isin(keys["새채무자키"])].copy()[법조치cols]

In [ ]:
# 빈값채우기 (빈 문자열은 NA로 먼저 바꿔줌 : 함수내부처리)
court = fill_groupwise(df=court, column="접수일")
court = fill_groupwise(df=court, column="종료일")
court = fill_groupwise(df=court, column="종국결과일")
court = fill_groupwise(df=court, column="종국결과")
court = fill_groupwise(df=court, column="진행결과")
court = fill_groupwise(df=court, column="확정일")
court = fill_groupwise(df=court, column="대상자구분")
court = fill_groupwise(df=court, column="대상자명")

In [ ]:
# 새채무자키, 사건번호, 관할법원, 최신화결과화면 순으로 정렬
court = court.sort_values(by=["새채무자키", "사건번호", "관할법원", "최신화결과화면"]).reset_index(drop=True)
# court에서 관할법원과 사건번호가 중복건인 경우는 제외
court_unique = court[~court.duplicated(subset=["관할법원", "사건번호"], keep="first")].copy()

In [ ]:
# court와 court_unique 엑셀파일로 출력
court.to_excel(join(path_write_dir, company+"_법조치조회_전체.xlsx"), index=False)
court_unique.to_excel(join(path_write_dir, company+"_법조치조회_중복제외.xlsx"), index=False)

### 법조치 내용 및 시효

In [37]:
####################################################
path_사건조회완료 = r"D:\3.자산\금감원\비정기 요청자료\260520\법조치조회_사건조회완료건.xlsx"
####################################################
사건조회완료 = pd.read_excel(path_사건조회완료, dtype=str) # 모두 문자열임에 유의

# 날짜형 변환
date_cols = ['확정일', '종료일', '종국결과일', '접수일','법조치시효']
for col in date_cols:
    사건조회완료[col] = pd.to_datetime(사건조회완료[col], errors='coerce')  # 변환 불가능한 값은 NaT 처리

print(len(사건조회완료))

# 법조치내용일 칼럼 추가
사건조회완료["법조치내용일"] = 사건조회완료[['확정일', '종료일', '종국결과일', '접수일']].max(axis=1)

12231


In [38]:
# 시효
# 새채무자키별 가장 나중 법조치시효
법조치시효 = 사건조회완료.groupby('새채무자키', as_index=False)['법조치시효'].max()
print(len(법조치시효))

4528


In [41]:
# 법조치 내용에서 제외할 거
#######################
매입전제외 = True
기준일이후제외 = True
기준일 = pd.to_datetime('2026-05-20')
#######################

법조치내용 = 사건조회완료[["새채무자키","법조치기간", "법조치분류",'법조치내용','법조치내용일','접수일','종국결과일','확정일']].copy()

if 매입전제외 : 
    # 매입후만 (매입전 제외)
    print("매입전 법조치건수", (법조치내용.법조치기간!="매입후").sum())
    법조치내용 = 법조치내용[법조치내용.법조치기간=="매입후"]
    print("매입후 법조치건수", len(법조치내용))

# 기준일 이후 접수건 2026-05-21 이후 접수건
if 기준일이후제외 : 
    print("기준일이후 법조치 접수건 제외전",len(법조치내용))
    법조치내용 = 법조치내용[법조치내용.접수일<=기준일]
    print("기준일이후 법조치 접수건 제외후",len(법조치내용))

매입전 법조치건수 6086
매입후 법조치건수 6145
기준일이후 법조치 접수건 제외전 6145
기준일이후 법조치 접수건 제외후 6114


In [ ]:
# 예시: 법조치내용 법조치내용
# 법조치내용 = pd.DataFrame(...)

def select_representative(group):
    # 법조치분류별 우선순위 로직
    if group.name[1] == "1.집행권원":
        # 확정일이 있는 경우 → 가장 나중
        if group['확정일'].notna().any():
            return group.loc[group['확정일'].idxmax()]
        else:
            return group.loc[group['접수일'].idxmax()]
    else:
        # 나머지 분류
        if group['종국결과일'].notna().any():
            return group.loc[group['종국결과일'].idxmax()]
        else:
            return group.loc[group['접수일'].idxmax()]


# 새채무자키 + 법조치분류 단위로 대표 행 선택
representatives = (
    법조치내용.groupby(['새채무자키','법조치분류'], group_keys=False)
      .apply(select_representative)
)


# 피벗테이블: 내용과 날짜를 함께
pivot_content = representatives.pivot(index='새채무자키',
                                      columns='법조치분류',
                                      values='법조치내용')

pivot_date = representatives.pivot(index='새채무자키',
                                   columns='법조치분류',
                                   values='법조치내용일')

# 날짜를 보기 좋게 문자열로 변환
pivot_date = pivot_date.applymap(lambda x: x.strftime('%Y-%m-%d') if pd.notna(x) else x)

# 두 피벗 병합
final_pivot = pivot_content.merge(pivot_date, left_index=True, right_index=True, suffixes=('', '_일'))


# 넘버링 붙은 분류명만 추출 (예: '1.집행권원', '2.보전처분' ...)
base_cols = [col for col in final_pivot.columns if not col.endswith('_일')]

# 자동으로 순서 리스트 생성
desired_order = []
for col in sorted(base_cols, key=lambda x: int(x.split('.')[0])):  # 앞 번호 기준 정렬
    desired_order.append(col)
    if f"{col}_일" in final_pivot.columns:
        desired_order.append(f"{col}_일")

# 순서 적용
final_pivot = final_pivot[desired_order]

# 리셋인덱스
final_pivot = final_pivot.reset_index()

print(len(final_pivot))
display(final_pivot.head())

법조치분류,새채무자키,1.집행권원,1.집행권원_일,2.보전처분,2.보전처분_일,3.재산명시,3.재산명시_일,4.채불,4.채불_일,5.채권압류,5.채권압류_일
0,n1001005,"소송 판결 확정(대여금, 양수금 등 이행의 소)",2019-12-21,NaN,NaT,NaN,NaT,NaN,NaT,NaN,NaT
1,n1001006,"소송 판결 확정(대여금, 양수금 등 이행의 소)",2025-05-16,NaN,NaT,NaN,NaT,NaN,NaT,NaN,NaT
2,n1001007,지급명령 확정,2021-01-15,NaN,NaT,NaN,NaT,NaN,NaT,채권 압류명령 확정(전부명령·추심명령 포함),2023-03-23
3,n1001015,지급명령 확정,2021-02-04,NaN,NaT,NaN,NaT,NaN,NaT,채권 압류명령 확정(전부명령·추심명령 포함),2021-04-07
4,n1001024,NaN,NaT,NaN,NaT,재산목록 열람 또는 재산조회,2019-08-12,NaN,NaT,채권 압류명령 확정(전부명령·추심명령 포함),2018-08-20


In [45]:
# 새채무자키열을 기준으로 keys와 병합
rst_법조치 = keys[["새채무자키","계좌키"]].merge(final_pivot, how='left')

# 법조치시효 병합
rst_법조치 = rst_법조치.merge(법조치시효, how='left')

# 법조치시효열 형태 정리
rst_법조치['법조치시효'] = rst_법조치['법조치시효'].dt.strftime('%Y-%m-%d')
# na는 빈문자열로
rst_법조치.fillna('', inplace=True)

rst_법조치

,새채무자키,계좌키,1.집행권원,1.집행권원_일,2.보전처분,2.보전처분_일,3.재산명시,3.재산명시_일,4.채불,4.채불_일,5.채권압류,5.채권압류_일,법조치시효
0,n1001206,200924167,,,,,,,,,채권 압류명령 확정(전부명령·추심명령 포함),2024-11-20,2100-01-01
1,n1049044,200950187,,,,,,,,,,,
2,n1047846,200949808,"소송 판결 확정(대여금, 양수금 등 이행의 소)",2024-07-26,,,,,,,,,2100-01-01
3,n1002718,200925982,,,,,,,,,,,
4,n1002720,200933851,,,,,,,,,,,
...,...,...,...,...,...,...,...,...,...,...,...,...,...
6814,n1033917,201028036,,,,,,,,,채권 압류명령 확정(전부명령·추심명령 포함),2022-11-08,2100-01-01
6815,n1033917,201028037,,,,,,,,,채권 압류명령 확정(전부명령·추심명령 포함),2022-11-08,2100-01-01
6816,n1029485,200946058,,,,,,,,,,,2100-01-01
6817,n1040698,200986220,지급명령 확정,2023-05-12,,,,,,,채권 압류명령 확정(전부명령·추심명령 포함),2026-02-19,2100-01-01


In [46]:
# 엑셀 출력
rst_법조치.to_excel(join(path_write_dir, "rst_"+company+"_법조치내용및시효정리.xlsx"), index=False)

### 입금

In [ ]:
if company == "솔림" : 
    deposit_ori = pd.read_parquet(join(r"D:\3.자산\전산 dataset\솔림\기간축적데이터\입금\parquet","솔림입금전체_160119~260531.parquet"))
else : 
# 대성                                          ########################################
    deposit_ori = pd.read_excel(join(path_write_dir,"입금조회새창_~20260520.xlsx"), dtype=deposit_dtype)
    # keys = pd.read_excel(join(path_write_dir,"새도약기금대상채권키리스트.xlsx"),dtype=str)

In [ ]:
# keys에 계좌키가 있는 입금건만 남기기
deposit = deposit_ori[deposit_ori["계좌키"].isin(keys["계좌키"])].copy()

# 입금구분, 기간 조건
입금cols = ["채무자키", "계좌키", "입금합계","입금일", "입금구분","입금자구분","입금자", "입금기간"]
cond1 = deposit.입금구분.isin(["신용회복","개인회생","약정분납","일반입금","임의변제","채권압류","배당금","유체동산","회생폐지","부동산경매","대위변제","파산배당","상속배당"])
deposit['입금기간'] = deposit['입금기간'].astype(str)
cond2 = ~(deposit.입금기간.str.contains(r"매입전|매각후|환매|회수제외|오류입금|환불", regex=True))  

deposit = deposit.loc[(cond1 & cond2)]
deposit = deposit[입금cols]
deposit

In [ ]:
import pandas as pd

# 기준일 정의
dates = {
    "기준일1": "2026-05-21",
    "기준일2": "2025-06-19",
    "기준일3": "2024-06-19",
    "기준일4": "2023-06-19",
    "기준일5": "2022-06-19",
    "기준일6": "2021-06-19",
    "기준일7": "2020-06-19",
    "기준일8": "2019-06-19",
}

# datetime 변환
for k, v in dates.items():
    dates[k] = pd.to_datetime(v)

deposit["입금일"] = pd.to_datetime(deposit["입금일"])

# 상환기간 판별 함수
def get_period(date):
    if dates["기준일2"] <= date < dates["기준일1"]:
        return "상환기간1"
    elif dates["기준일3"] <= date < dates["기준일2"]:
        return "상환기간2"
    elif dates["기준일4"] <= date < dates["기준일3"]:
        return "상환기간3"
    elif dates["기준일5"] <= date < dates["기준일4"]:
        return "상환기간4"
    elif dates["기준일6"] <= date < dates["기준일5"]:
        return "상환기간5"
    elif dates["기준일7"] <= date < dates["기준일6"]:
        return "상환기간6"
    elif dates["기준일8"] <= date < dates["기준일7"]:
        return "상환기간7"
    elif date < dates["기준일8"]:
        return "상환기간8"
    else:
        return None

# 새로운 열 생성
deposit["상환기간"] = deposit["입금일"].apply(get_period)

In [ ]:
# 계좌키별로 상환기간1 ~ 상환기간8까지 입금합계 구하고, keys의 계좌키와 merge
period_sum = deposit.groupby(["계좌키", "상환기간"])["입금합계"].sum().unstack(fill_value=0).reset_index()
period_sum = period_sum.rename_axis(None, axis=1)  # 상환기간 열 이름에서 계층 제거
# 총합계열 만들기
period_sum["총합계"] = period_sum.iloc[:, 1:].sum(axis=1)
# 총합계열을 계좌키 옆 열로(2번째열) 이동
period_sum = period_sum[['계좌키', '총합계'] + [col for col in period_sum.columns if col != '계좌키' and col != '총합계']]

# keys와 merge
result_deposit = keys[["채무자키","계좌키"]].merge(period_sum, on="계좌키", how="left")
# na는 0으로 바꿔주기
result_deposit = result_deposit.fillna(0)

In [ ]:
result_deposit.to_excel(join(path_write_dir, company+"_계좌별_상환기간별_입금합계.xlsx"), index=False)

### 조정